# 03. Feature Engineering

This notebook prepares the data for modeling by performing feature engineering, including One-Hot Encoding and column dropping.

In [10]:
# Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [11]:
# Load Interim Data
train_df = pd.read_csv('interim_train.csv')
test_df = pd.read_csv('interim_test.csv')
store_df = pd.read_csv('interim_store.csv')

print("Data loaded successfully.")

Data loaded successfully.


In [12]:
# Merge store data
train_df = pd.merge(train_df, store_df, on='Store')
test_df = pd.merge(test_df, store_df, on='Store')

In [13]:
# Drop Year and Month (as per original logic)
train_df.drop(["Year", "Month"], axis=1, inplace=True)
test_df.drop(["Year", "Month"], axis=1, inplace=True)

# Also drop DateStr if it exists
if 'DateStr' in train_df.columns:
    train_df.drop(['DateStr'], axis=1, inplace=True)
if 'DateStr' in test_df.columns:
    test_df.drop(['DateStr'], axis=1, inplace=True)

In [14]:
# Dummy Variables for DayOfWeek
day_dummies_train = pd.get_dummies(train_df['DayOfWeek'], prefix='Day')
day_dummies_train.drop(['Day_7'], axis=1, inplace=True)

day_dummies_test = pd.get_dummies(test_df['DayOfWeek'], prefix='Day')
day_dummies_test.drop(['Day_7'], axis=1, inplace=True)

train_df = train_df.join(day_dummies_train)
test_df = test_df.join(day_dummies_test)

train_df.drop(['DayOfWeek'], axis=1, inplace=True)
test_df.drop(['DayOfWeek'], axis=1, inplace=True)

In [15]:
# Filter Closed Stores and Drop Columns
# remove all rows(store,date) that were closed
train_df = train_df[train_df["Open"] != 0]

# drop unnecessary columns
train_df.drop(["Open", "Customers", "Date"], axis=1, inplace=True)

# For test_df, we keep closed stores for submission purposes effectively, 
# but the original code removed them and filled with 0 later.
# "closed_store_ids = test_df["Id"][test_df["Open"] == 0].values"
# "test_df = test_df[test_df["Open"] != 0]"

closed_store_ids = test_df["Id"][test_df["Open"] == 0].values
test_df = test_df[test_df["Open"] != 0]
test_df.drop(['Open', 'Date'], axis=1, inplace=True)

# Save closed_store_ids for later use in modeling notebook if needed (though mapped to 0)
cleaned_test_ids = test_df["Id"]

In [16]:
# Determine features for model
# We need to handle categorical variables like StateHoliday, StoreType, Assortment
# Original code used specific mapping or just didn't handle them explicitly in the provided snippet beyond mapping StateHoliday 0 to "0".
# Wait, the provided snippet has:
# rossmann_df["StateHoliday"] = rossmann_df["StateHoliday"].map({0: 0, "0": 0, "a": 1, "b": 1, "c": 1})
# Let's add that map.

train_df["StateHoliday"] = train_df["StateHoliday"].map({0: 0, "0": 0, "a": 1, "b": 1, "c": 1})
test_df["StateHoliday"] = test_df["StateHoliday"].map({0: 0, "0": 0, "a": 1, "b": 1, "c": 1})

# Check other categoricals: StoreType, Assortment, PromoInterval
# The original code doesn't seem to encode them in the snippet provided?
# Snippet: "X_train = store.drop(["Sales","Store"],axis=1)"
# If StoreType is string, sklearn LinearRegression might fail. 
# But the original code ran `lreg.fit(X_train, Y_train)`. 
# Maybe they were dropped? No, snippet says `rossmann_df.drop(["Open","Customers", "Date"], ...)`
# Maybe pandas get_dummies was used on them? 
# Ah, the snippet provided earlier: `day_dummies_rossmann`... 
# There is no explicit encoding for StoreType/Assortment in the snippet I have.
# However, in `02`, I see `train_store['StoreType'] = train_store['StoreType'].astype(str)`.
# I should probably encode them to be safe, or drop them if they are not numeric.
# LinearRegression needs numeric input. 
# I recall standard Rossmann usually maps 'a','b','c','d' to 1,2,3,4 or dummies. 
# I will use label encodings or dummies for StoreType and Assortment to ensure models run.

# Mapping for StoreType and Assortment
train_df['StoreType'] = train_df['StoreType'].map({'a':1, 'b':2, 'c':3, 'd':4})
train_df['Assortment'] = train_df['Assortment'].map({'a':1, 'b':2, 'c':3})

test_df['StoreType'] = test_df['StoreType'].map({'a':1, 'b':2, 'c':3, 'd':4})
test_df['Assortment'] = test_df['Assortment'].map({'a':1, 'b':2, 'c':3})

# PromoInterval is also object. 
# Let's drop it or encode it. Original code didn't specify. I'll drop it to avoid errors.
if 'PromoInterval' in train_df.columns:
    train_df.drop('PromoInterval', axis=1, inplace=True)
if 'PromoInterval' in test_df.columns:
    test_df.drop('PromoInterval', axis=1, inplace=True)

In [17]:
# Save Processed Data
train_df.to_csv('processed_train.csv', index=False)
test_df.to_csv('processed_test.csv', index=False)

print("Processed files saved: processed_train.csv, processed_test.csv")

Processed files saved: processed_train.csv, processed_test.csv
